In [3]:
import warnings
import random
import joblib
import pandas as pd
import numpy as np

# For reproducibility
random.seed(42)
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

# -------------------------
# RDKit Imports
# -------------------------
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem  # Using the older method (will issue a deprecation warning)
warnings.filterwarnings("ignore", message=".*please use MorganGenerator.*")

# -------------------------
# Scikit-learn and other ML libraries
# -------------------------
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, make_scorer
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Neural network via SciKeras
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# -------------------------
# 1. Data Loading (CSV) & Filtering
# -------------------------
def load_data(file_path="iupac_high-confidence_v2_2.csv", 
              smiles_col="SMILES", 
              pka_col="pka_value",
              pka_type_col="pka_type"):
    """
    Loads data from the CSV file and returns a DataFrame with SMILES, pKa, and pKa type.
    """
    df = pd.read_csv(file_path)
    print("Data columns found:", df.columns.tolist())
    
    # Keep only the columns we care about
    df = df[[smiles_col, pka_col, pka_type_col]].dropna(subset=[smiles_col, pka_col, pka_type_col])
    
    # Convert pKa values to numeric
    df[pka_col] = pd.to_numeric(df[pka_col], errors='coerce')
    df = df.dropna(subset=[pka_col]).reset_index(drop=True)
    
    print(f"Loaded {len(df)} valid rows from {file_path}")
    return df

def filter_data_by_pka_type(df, target_type="pKa", pka_type_col="pka_type"):
    """
    Filter the DataFrame to only include rows with the specified pKa type.
    If no rows match, the full dataset is returned with a warning.
    """
    # Print unique pKa types for your inspection
    unique_types = df[pka_type_col].unique()
    print("Unique pKa types in the dataset:", unique_types)
    
    filtered_df = df[df[pka_type_col].str.strip().str.upper() == target_type.upper()].reset_index(drop=True)
    if len(filtered_df) == 0:
        print(f"Warning: No rows found with pKa type '{target_type}'. Using the full dataset instead.")
        return df
    else:
        print(f"Filtered data to {len(filtered_df)} rows with pKa type '{target_type}'.")
        return filtered_df

# -------------------------
# 2. Feature Extraction using RDKit
# -------------------------
def get_morgan_fingerprint(smiles, radius=2, nBits=2048):
    """
    Generate a Morgan fingerprint using AllChem.GetMorganFingerprintAsBitVect.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def get_maccs_keys(smiles):
    """
    Generate MACCS Keys fingerprint from a SMILES string.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    from rdkit.Chem import MACCSkeys
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def featurize_data(data, 
                   smiles_col="SMILES", 
                   fingerprint_type="morgan"):
    """
    Generate molecular fingerprints for all molecules in the dataset.
    """
    if fingerprint_type.lower() == "morgan":
        data["Fingerprint"] = data[smiles_col].apply(get_morgan_fingerprint)
    elif fingerprint_type.lower() == "maccs":
        data["Fingerprint"] = data[smiles_col].apply(get_maccs_keys)
    else:
        raise ValueError("Unsupported fingerprint type. Choose 'morgan' or 'maccs'.")
    
    data = data[data["Fingerprint"].notnull()].reset_index(drop=True)
    X = np.array(data["Fingerprint"].tolist())
    return X, data

# -------------------------
# 3. Define the Neural Network Model Builder
# -------------------------
def build_nn_model(input_dim):
    """
    Build a simple neural network for regression.
    """
    model = Sequential()
    model.add(Dense(128, activation='relu', input_dim=input_dim))
    model.add(Dropout(0.2))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1))  # Output layer for regression
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    return model

# -------------------------
# 4. Model Training & Evaluation
# -------------------------
def train_and_evaluate_models(X, y):
    """
    Train and evaluate various regression models using 5-fold cross-validation.
    """
    models = {
        "Linear Regression": LinearRegression(),
        "Elastic Net": ElasticNet(random_state=42),
        "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
        "Random Forest": RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1),
        "Support Vector Machine": SVR(),
        "XGBoost": XGBRegressor(random_state=42, objective='reg:squarederror'),
        "LightGBM": LGBMRegressor(random_state=42),
        "CatBoost": CatBoostRegressor(random_state=42, verbose=0),
        "Neural Network": KerasRegressor(
            model=build_nn_model,
            model__input_dim=X.shape[1],
            epochs=50,
            batch_size=32,
            verbose=0,
            random_state=42
        )
    }
    
    scoring = {
        "rmse": make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": "r2",
        "mae": make_scorer(mean_absolute_error)
    }
    
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    results = {}
    
    for name, model in models.items():
        print(f"\nEvaluating {name}...")
        cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
        mean_rmse = np.mean(cv_results["test_rmse"])
        mean_r2 = np.mean(cv_results["test_r2"])
        mean_mae = np.mean(cv_results["test_mae"])
        results[name] = {"RMSE": mean_rmse, "R2": mean_r2, "MAE": mean_mae}
        print(f"{name}: RMSE = {mean_rmse:.4f}, R2 = {mean_r2:.4f}, MAE = {mean_mae:.4f}")
    
    ranked_models = sorted(results.items(), key=lambda x: x[1]["RMSE"])
    print("\nModel Ranking (based on RMSE):")
    for rank, (model_name, metrics) in enumerate(ranked_models, start=1):
        print(f"{rank}. {model_name}: RMSE = {metrics['RMSE']:.4f}, R2 = {metrics['R2']:.4f}, MAE = {metrics['MAE']:.4f}")
    
    return results, models, ranked_models

# -------------------------
# 5. Final Output & Saving the Best Model
# -------------------------
def train_best_model(X, y, best_model_name, models):
    """
    Retrain the best-performing model on the full dataset and save it.
    """
    print(f"\nRetraining the best model: {best_model_name} on the full dataset...")
    best_model = models[best_model_name]
    best_model.fit(X, y)
    
    # Save the model for future use
    if best_model_name == "Neural Network":
        best_model.model_.save("best_model_neural_network.h5")
        print("Neural Network saved as 'best_model_neural_network.h5'")
    else:
        filename = f"best_model_{best_model_name.replace(' ', '_')}.pkl"
        joblib.dump(best_model, filename)
        print(f"{best_model_name} saved as '{filename}'")
    
    return best_model

# -------------------------
# 6. Main Function
# -------------------------
def main():
    # Step 1: Load Data from CSV
    csv_file = "iupac_high-confidence_v2_2.csv"  # Ensure this file is in your working directory
    smiles_col = "SMILES"
    pka_col = "pka_value"
    pka_type_col = "pka_type"
    data = load_data(file_path=csv_file, smiles_col=smiles_col, pka_col=pka_col, pka_type_col=pka_type_col)
    
    # Optional: Filter Data by a Specific pKa Type
    target_pka_type = "pKa"  # Change this value to your desired type (e.g., "pKaH", "pKb", etc.)
    filtered_data = filter_data_by_pka_type(data, target_type=target_pka_type, pka_type_col=pka_type_col)
    
    # Step 2: Feature Extraction
    fingerprint_type = "morgan"  # Or "maccs" if preferred
    X, filtered_data = featurize_data(filtered_data, smiles_col=smiles_col, fingerprint_type=fingerprint_type)
    y = filtered_data[pka_col].values
    print(f"Feature matrix shape: {X.shape}")
    print(f"Target vector shape: {y.shape}")
    
    # Step 3: Model Training & Evaluation
    results, models, ranked_models = train_and_evaluate_models(X, y)
    best_model_name = ranked_models[0][0]
    best_metrics = ranked_models[0][1]
    print(f"\nBest performing model: {best_model_name}")
    print(f"Performance: RMSE = {best_metrics['RMSE']:.4f}, R2 = {best_metrics['R2']:.4f}, MAE = {best_metrics['MAE']:.4f}")
    
    # Step 4: Final Training and Saving the Best Model
    best_model = train_best_model(X, y, best_model_name, models)
    
    # Step 5: Summary of Findings
    print("\nSummary of Findings:")
    print("Performance metrics from 5-fold cross-validation:")
    for name, metrics in results.items():
        print(f"{name}: RMSE = {metrics['RMSE']:.4f}, R2 = {metrics['R2']:.4f}, MAE = {metrics['MAE']:.4f}")
    print(f"\nThe best performing model is: {best_model_name}")
    print("This model has been retrained on the full dataset and saved for future use.")

if __name__ == "__main__":
    main()


Data columns found: ['unique_ID', 'SMILES', 'InChI', 'pka_type', 'pka_value', 'T', 'remarks', 'method', 'assessment', 'ref', 'ref_remarks', 'entry_remarks', 'original_IUPAC_names', 'name_contributors', 'num_name_contributors', 'original_IUPAC_nicknames', 'source', 'pressure', 'acidity_label', 'original_T', 'solvent']
Loaded 24028 valid rows from iupac_high-confidence_v2_2.csv
Unique pKa types in the dataset: ['pKaH1' 'pKb' 'pKaH2' 'pKaH3' 'pKa1' 'pKaH4' 'pKaH5' 'pKa2' 'pKb1' 'pKb2'
 'pKaH1(50% ethanol)' 'pKa3' 'pKaH(C+)' 'pKaH(A+)' 'pKaH(A+A+)'
 'pKaH(C+A+)' 'pKaH(C+A+A+)' 'pKa1(SO2NH)' 'pKa1(NH)' 'pKb3'
 'pKaH1(predict)' 'pKaH2(anhydrous)' 'pKaH(eq)' 'pKaH1(hydrated)'
 'pKaH(neut)' 'pKa1(anhydrous)' 'pKa2(anhydrous)' 'pKa1(hydrated)'
 'pKa2(hydrated)' 'pKaH1(hydrate)' 'pKa1(eq)' 'pKaH2(hydrated)'
 'pKa1(hydrate)' 'pKaH(excited)' 'pKaH2(50% ethanol)'
 'pKaH(dihydrochloride)' 'pKa1(50% ethanol)' 'pKaH(unknown)'
 'Nc1cccc2cccc(S(=O)(=O)O)c12' 'pKb(2nd-conjugate)' 'pKa4' 'pKa5'
 'pK(unkno

[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerator
[20:17:13] DEPRECATION WARNING: please use MorganGenerat

Feature matrix shape: (24028, 2048)
Target vector shape: (24028,)

Evaluating Linear Regression...
Linear Regression: RMSE = 8137764038.9209, R2 = -19580616487000797184.0000, MAE = 203338225.6072

Evaluating Elastic Net...
Elastic Net: RMSE = 4.0636, R2 = -0.0002, MAE = 3.2743

Evaluating K-Nearest Neighbors...
K-Nearest Neighbors: RMSE = 3.3429, R2 = 0.3230, MAE = 2.2760

Evaluating Random Forest...
Random Forest: RMSE = 3.2320, R2 = 0.3670, MAE = 2.1552

Evaluating Support Vector Machine...
Support Vector Machine: RMSE = 3.1883, R2 = 0.3842, MAE = 2.0947

Evaluating XGBoost...
XGBoost: RMSE = 3.0752, R2 = 0.4271, MAE = 2.2172

Evaluating LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020598 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3724
[LightGBM] [Info] Number of data points in the train set: 19222, number of used 

C:\Users\Fahad\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Fahad\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Fahad\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs

Neural Network: RMSE = 3.3165, R2 = 0.3334, MAE = 2.2318

Model Ranking (based on RMSE):
1. CatBoost: RMSE = 3.0428, R2 = 0.4391, MAE = 2.2191
2. LightGBM: RMSE = 3.0487, R2 = 0.4369, MAE = 2.2210
3. XGBoost: RMSE = 3.0752, R2 = 0.4271, MAE = 2.2172
4. Support Vector Machine: RMSE = 3.1883, R2 = 0.3842, MAE = 2.0947
5. Random Forest: RMSE = 3.2320, R2 = 0.3670, MAE = 2.1552
6. Neural Network: RMSE = 3.3165, R2 = 0.3334, MAE = 2.2318
7. K-Nearest Neighbors: RMSE = 3.3429, R2 = 0.3230, MAE = 2.2760
8. Elastic Net: RMSE = 4.0636, R2 = -0.0002, MAE = 3.2743
9. Linear Regression: RMSE = 8137764038.9209, R2 = -19580616487000797184.0000, MAE = 203338225.6072

Best performing model: CatBoost
Performance: RMSE = 3.0428, R2 = 0.4391, MAE = 2.2191

Retraining the best model: CatBoost on the full dataset...
CatBoost saved as 'best_model_CatBoost.pkl'

Summary of Findings:
Performance metrics from 5-fold cross-validation:
Linear Regression: RMSE = 8137764038.9209, R2 = -19580616487000797184.0000, M

In [5]:
#Testing

In [17]:
import sys
import numpy as np
import pubchempy as pcp
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import pickle  # Import pickle

def get_smiles_from_iupac(iupac_name):
    compounds = pcp.get_compounds(iupac_name, 'name')
    if compounds:
        return compounds[0].canonical_smiles
    else:
        raise ValueError("Compound not found in PubChem.")

def smiles_to_fingerprint(smiles, radius=2, fpSize=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Could not generate a molecule from the SMILES string.")
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fpSize)
    fp = generator.GetFingerprint(mol)
    fp_array = np.zeros((fpSize,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, fp_array)
    return fp_array.reshape(1, -1)

def main():
    # Step 1: Get IUPAC name input from the user
    iupac_name = input("Enter the IUPAC name of the compound: ").strip()
    
    try:
        # Step 2: Convert IUPAC name to SMILES using PubChem
        smiles = get_smiles_from_iupac(iupac_name)
        print(f"Converted SMILES: {smiles}")
    except Exception as e:
        print("Error converting IUPAC name to SMILES:", e)
        sys.exit(1)
    
    try:
        # Step 3: Generate the Morgan fingerprint
        fingerprint = smiles_to_fingerprint(smiles)
        print("Generated fingerprint (as numpy array):")
        print(fingerprint)
    except Exception as e:
        print("Error generating fingerprint:", e)
        sys.exit(1)
    
    # Step 4: Load the model using pickle
    try:
        with open("best_model_CatBoost.pkl", "rb") as f:
            model = pickle.load(f)
    except Exception as e:
        print("Error loading the prediction model with pickle:", e)
        sys.exit(1)
    
    try:
        # Step 5: Predict the pKa value using the model
        predicted_pka = model.predict(fingerprint)
        predicted_value = predicted_pka[0] if isinstance(predicted_pka, (list, np.ndarray)) else predicted_pka
        print(f"Predicted pKa value: {predicted_value}")
    except Exception as e:
        print("Error during prediction:", e)
        sys.exit(1)

if __name__ == "__main__":
    main()

Enter the IUPAC name of the compound:  methanol


Converted SMILES: CO
Generated fingerprint (as numpy array):
[[0 0 0 ... 0 0 0]]
Predicted pKa value: 6.028216180650861
